# Class 5: Probability

## Scoring Asset Probability of Flooding

In this class, we'll learn how to assess **the probability that an asset (parcel and building) will be affected by flooding** based on which flood zone it intersects.

### Learning Objectives
- Understand flood zone categories and what they mean for risk
- Learn how to perform spatial joins in Python/GeoPandas
- Score parcels by flood probability using a hierarchy system
- Create risk visualizations and statistics
- Understand GIS equivalents in QGIS and ArcGIS Pro

### What is Probability in This Context?
**Probability** answers the question: *"How likely is flooding to occur at this location in any given year?"*

Flood zones are defined by the Federal Emergency Management Agency (FEMA) based on statistical analysis of historical flood data. Each zone represents a specific probability level:
- **Floodway**: The highest probability — water MUST flow through this area when flooding occurs
- **100-year Floodplain**: 1% annual chance of flooding in any given year
- **500-year Floodplain**: 0.2% annual chance of flooding in any given year

### Key Concept: Flood Zone Hierarchy
Some parcels may intersect **multiple flood zones**. When this happens, we use the **highest-risk zone** for scoring:
- A parcel touching both the 100-year floodplain AND the floodway gets scored as "Floodway" (highest risk)
- A parcel touching both the 100-year and 500-year zones gets scored as "100-year" (higher risk)

**This hierarchy is critical** — we want to capture the true maximum risk that the asset faces.

## Step 1: Setup & Install Libraries

Before we begin, we'll mount Google Drive and install the necessary GIS libraries. These libraries allow us to:
- **geopandas**: Work with geographic data (shapefiles, GeoPackages, etc.)
- **folium**: Create interactive maps
- **pandas & numpy**: Analyze and manipulate data
- **matplotlib**: Create visualizations

In [ ]:
# === ENVIRONMENT SETUP ===
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/MSER_510_VULNERABILTY'
    print("Google Drive connected!")
except Exception:
    BASE_DIR = './'
    print("Running locally.")

DATA_DIR = os.path.join(BASE_DIR, 'data')
OUTPUT_DIR = os.path.join(BASE_DIR, 'outputs')

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# GeoPackage file paths for chained loading
INPUT_GPKG = os.path.join(DATA_DIR, 'class_4_vulnerability.gpkg')  # Load from Class 4
OUTPUT_GPKG = os.path.join(DATA_DIR, 'class_5_probability.gpkg')  # Save to Class 5
CLASS0_GPKG = os.path.join(DATA_DIR, 'vulnerability_risk_data.gpkg')  # Fallback for base layers

print(f"  Base directory: {BASE_DIR}")
print(f"  Data directory: {DATA_DIR}")
print(f"  Output directory: {OUTPUT_DIR}")

In [ ]:
# Install required libraries
!pip install geopandas fiona shapely pyproj requests folium seaborn contextily --quiet

# Import libraries
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import folium
import os
import warnings
warnings.filterwarnings('ignore')

## Step 2: Load Data from GeoPackage

We'll load two layers:
1. **parcels**: The parcels from previous classes (with location, building characteristics, and values)
2. **flood_zones**: Boundaries of different FEMA flood zones

The flood_zones layer should have a `flood_category` field with three possible values:
- 'Floodway'
- '100-year'
- '500-year'

Let's examine the data to make sure we understand its structure.

In [ ]:
# List all layers in the GeoPackage
import fiona
print("Layers in GeoPackage:")
layers = fiona.listlayers(INPUT_GPKG)
print(layers)

In [ ]:
# Load the parcels layer from INPUT GeoPackage (from Class 4)
parcels = gpd.read_file(INPUT_GPKG, layer='parcels')
print(f"✓ Loaded {len(parcels)} parcels from Class 4")
print(f"\nColumns: {parcels.columns.tolist()}")
print(f"\nFirst few rows:")
print(parcels.head())

In [ ]:
# Load the flood zones layer from INPUT GeoPackage (from Class 4)
flood_zones = gpd.read_file(INPUT_GPKG, layer='flood_zones')
print(f"✓ Loaded {len(flood_zones)} flood zone features")
print(f"\nFlood zone categories:")
print(flood_zones['flood_category'].value_counts())
print(f"\nFirst few rows:")
print(flood_zones[['FLD_ZONE', 'ZONE_SUBTY', 'flood_category']].head())

## Step 3: Understanding the Flood Zone Hierarchy

Before we score the parcels, let's think about what each flood zone means:

### Floodway (Highest Risk = 3)
The floodway is the channel of the river plus adjacent areas that must be kept **completely free of obstructions**. Water MUST flow through the floodway during flood events. If a building is in the floodway, it will almost certainly experience flooding.

### 100-year Floodplain (Medium Risk = 2)
Also called the "base flood elevation," this is the area that has a **1% chance of flooding in any given year**. Over 30 years, there's roughly a 26% chance of experiencing a flood. This is the regulatory standard used by FEMA.

### 500-year Floodplain (Lower Risk = 1)
This area has a **0.2% annual chance** of flooding (or 20% over 100 years). While floods are less frequent here, they can still occur and be severe.

### Outside All Zones (No Risk = 0)
Parcels outside all flood zones have no recorded flood risk and receive a score of 0.

### The Hierarchy Rule
When a parcel intersects multiple zones, we keep the **highest-priority category**:
- Floodway (3) > 100-year (2) > 500-year (1) > No zone (0)

This ensures we capture the true maximum risk that the asset faces.

## Step 4: Spatial Join — Find Which Flood Zones Each Parcel Intersects

A **spatial join** is a GIS operation that connects data from two layers based on their geographic location. In this case, we're asking: "For each parcel, which flood zones does it intersect?"

**How it works:**
1. For each parcel, check if it overlaps with any flood zone
2. If it does, record that flood zone's information (in this case, the `flood_category`)
3. If a parcel overlaps multiple zones, we'll get multiple records — we'll then keep only the highest-risk zone

Let's perform the spatial join:

In [ ]:
# Drop flood_category from parcels if it exists (carried forward from earlier classes)
# so the spatial join doesn't create _left/_right suffixes
if 'flood_category' in parcels.columns:
    parcels = parcels.drop(columns=['flood_category'])

# Spatial join: parcels with flood zones
# We use 'intersects' predicate to find parcels that touch or overlap flood zones
joined = gpd.sjoin(
    parcels,
    flood_zones[['geometry', 'flood_category']],
    how='left',  # Keep all parcels, even those not in any flood zone
    predicate='intersects'
)

print(f"Result of spatial join: {len(joined)} records")
print(f"(May be more than {len(parcels)} if some parcels intersect multiple zones)")
print(f"\nColumns after join:")
print(joined.columns.tolist())
print(f"\nFirst few rows:")
print(joined.head(10))

## Step 5: Handle Parcels in Multiple Flood Zones

When we performed the spatial join, some parcels may appear multiple times if they intersect more than one flood zone. For example, a parcel touching both the floodway and the 100-year floodplain will have two rows.

We need to keep only the **highest-risk zone** for each parcel. We do this by:
1. Creating a numeric priority: Floodway=3, 100-year=2, 500-year=1, None=0
2. For each parcel, finding the maximum priority value
3. Using that to assign the final probability score

This ensures that a parcel in multiple zones gets scored by its highest risk.

In [ ]:
# Create a mapping from flood category to numeric priority
flood_priority = {
    'Floodway': 3,
    '100-year': 2,
    '500-year': 1
}

# Map the flood category to its numeric priority
joined['flood_priority'] = joined['flood_category'].map(flood_priority).fillna(0)

print("Sample of joined data with priorities:")
print(joined[['parno', 'flood_category', 'flood_priority']].head(15))

In [ ]:
# Group by parcel and keep only the MAXIMUM priority for each
# This handles the case where a parcel intersects multiple zones

# Group by the parcel ID and get the maximum priority
max_priority_per_parcel = joined.groupby('parno')['flood_priority'].max()

print(f"Number of unique parcels: {len(max_priority_per_parcel)}")
print(f"\nDistribution of probability scores:")
print(max_priority_per_parcel.value_counts().sort_index(ascending=False))

## Step 6: Add Probability Score to Parcels (Separately for Each Asset Group)

We assign probability scores to all parcels, then create separate columns
`probability_a1` and `probability_a2` — only parcels in each asset group get scored.

In [ ]:
# Create a new column in parcels for probability
# Start with 0 for all parcels (those not in any flood zone)
parcels['probability'] = 0

# Update with the maximum priority for parcels that intersect flood zones
for parno in max_priority_per_parcel.index:
    parcels.loc[parcels['parno'] == parno, 'probability'] = int(max_priority_per_parcel[parno])

# Ensure asset flags exist
for col in ['is_asset_1', 'is_asset_2']:
    if col not in parcels.columns:
        print(f"WARNING: {col} not found — please ensure Class 1 was run first")

# Create separate probability columns per asset group
# Only parcels in the asset group get scored; others get 0
parcels['probability_a1'] = parcels['probability'].where(parcels['is_asset_1'] == 1, 0).astype(int)
parcels['probability_a2'] = parcels['probability'].where(parcels['is_asset_2'] == 1, 0).astype(int)

print("Probability scores calculated for both asset groups!")
for col, label in [('probability_a1', 'Asset 1'), ('probability_a2', 'Asset 2')]:
    asset_flag = 'is_asset_1' if 'a1' in col else 'is_asset_2'
    in_group = parcels[parcels[asset_flag] == 1]
    print(f"\n{label} ({len(in_group):,} parcels):")
    for score in sorted(in_group[col].unique(), reverse=True):
        count = (in_group[col] == score).sum()
        pct = 100 * count / len(in_group)
        plabel = {3: 'Floodway (High)', 2: '100-year (Medium)', 1: '500-year (Low)', 0: 'No Risk'}.get(score, '?')
        print(f"  {plabel} ({score}): {count:,} ({pct:.1f}%)")

## Step 7: Calculate Risk Statistics

Now let's generate summary statistics showing how many parcels fall into each risk category, and the total values at risk.

**Key questions we're answering:**
- How many parcels are in each risk category?
- What percentage of all parcels fall into each category?
- What is the total property and structure value at risk in each category?

This helps community leaders understand the scope of flood risk for planning and mitigation efforts.

In [ ]:
# Summary statistics — per asset group
risk_levels = {3: 'Floodway (High)', 2: '100-year (Medium)', 1: '500-year (Low)', 0: 'No Risk'}

for asset_label, prob_col, flag_col in [
    ('Asset 1', 'probability_a1', 'is_asset_1'),
    ('Asset 2', 'probability_a2', 'is_asset_2'),
]:
    subset = parcels[parcels[flag_col] == 1].copy()
    summary_stats = []
    for score in sorted(subset[prob_col].unique(), reverse=True):
        scored = subset[subset[prob_col] == score]
        count = len(scored)
        pct = (count / len(subset)) * 100 if len(subset) > 0 else 0
        risk_name = risk_levels.get(score, 'Unknown')

        prop_value = scored['property_value'].sum() if 'property_value' in scored.columns else 0
        struct_value = scored['structure_value'].sum() if 'structure_value' in scored.columns else 0

        summary_stats.append({
            'Risk Level': risk_name,
            'Score': score,
            'Parcel Count': count,
            'Percentage': f'{pct:.1f}%',
            'Property Value': f'${prop_value:,.0f}',
            'Structure Value': f'${struct_value:,.0f}',
        })

    summary_df = pd.DataFrame(summary_stats)
    print(f"\nProbability Score Summary — {asset_label} ({len(subset):,} parcels):")
    print(summary_df.to_string(index=False))


## Step 8: Visualize Probability Scores — Asset 1 vs Asset 2

Side-by-side maps showing the probability score for each asset group.
Parcels are colored by their flood zone score (Floodway = 3, 100-year = 2, 500-year = 1).

In [ ]:
import contextily as ctx
from matplotlib.patches import Patch
import matplotlib.gridspec as gridspec

# Reproject to Web Mercator for basemap tiles
parcels_wm = parcels.to_crs(epsg=3857)

# Create figure with two map panels side-by-side + shared legend below
fig = plt.figure(figsize=(18, 12))
gs = gridspec.GridSpec(2, 2, height_ratios=[10, 1.2], hspace=0.05, wspace=0.05)
ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])
ax_legend = fig.add_subplot(gs[1, :])

probability_colors = {1: '#B4D4E7', 2: '#8FABBE', 3: '#2B5797'}

for ax, col, asset_flag, title in [
    (ax1, 'probability_a1', 'is_asset_1', 'Probability — Asset 1'),
    (ax2, 'probability_a2', 'is_asset_2', 'Probability — Asset 2'),
]:
    # Layer 1: All parcels - thin grey borders
    parcels_wm.plot(ax=ax, facecolor='none', edgecolor='#888888', linewidth=0.3)

    # Layer 2: Probability scores for this asset group
    in_group = parcels_wm[parcels_wm[asset_flag] == 1]
    for score in [3, 2, 1]:
        subset = in_group[in_group[col] == score]
        if len(subset) > 0:
            subset.plot(ax=ax, facecolor=probability_colors[score], edgecolor='none', alpha=0.85)

    # Layer 3: Buildings
    try:
        buildings_layer = gpd.read_file(CLASS0_GPKG, layer='buildings')
        buildings_wm = buildings_layer.to_crs(epsg=3857)
        buildings_wm.plot(ax=ax, facecolor='#3D3D3D', edgecolor='#2a2a2a', linewidth=0.1, alpha=0.7)
    except Exception:
        pass

    # Layer 4: Flood zones
    try:
        flood_zones_full = gpd.read_file(CLASS0_GPKG, layer='flood_zones')
        flood_wm = flood_zones_full.to_crs(epsg=3857)
        flood_fill = {'Floodway': '#2B5797', '100-year': '#8FABBE', '500-year': '#B4D4E7'}
        for flood_type in ['500-year', '100-year', 'Floodway']:
            flood_subset = flood_wm[flood_wm['flood_category'] == flood_type]
            if len(flood_subset) > 0:
                flood_subset.plot(ax=ax, facecolor=flood_fill.get(flood_type, '#B4D4E7'),
                                edgecolor='none', alpha=0.3)
    except Exception:
        pass

    ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, zoom='auto')
    ax.set_axis_off()
    ax.set_title(title, fontsize=14, fontweight='bold', pad=10)

# Shared legend
ax_legend.set_axis_off()
legend_elements = [
    Patch(facecolor='#2B5797', edgecolor='none', label='Floodway (3)'),
    Patch(facecolor='#8FABBE', edgecolor='none', label='100-year (2)'),
    Patch(facecolor='#B4D4E7', edgecolor='none', label='500-year (1)'),
    Patch(facecolor='none', edgecolor='none', label=''),
    Patch(facecolor='#3D3D3D', edgecolor='#2a2a2a', label='Buildings'),
    Patch(facecolor='#2B5797', edgecolor='none', alpha=0.3, label='Floodway Zone'),
    Patch(facecolor='#8FABBE', edgecolor='none', alpha=0.3, label='100-year Zone'),
    Patch(facecolor='#B4D4E7', edgecolor='none', alpha=0.3, label='500-year Zone'),
    Patch(facecolor='none', edgecolor='#888888', linewidth=0.5, label='Parcels'),
]
ax_legend.legend(handles=legend_elements, loc='center', ncol=4, fontsize=9,
                frameon=True, facecolor='white', edgecolor='#cccccc',
                handlelength=1.5, handletextpad=0.5, columnspacing=1.5)

# Export
OUTPUT_DIR = os.path.join(BASE_DIR, 'outputs')
os.makedirs(OUTPUT_DIR, exist_ok=True)
png_path = os.path.join(OUTPUT_DIR, 'probability_map.png')
plt.savefig(png_path, dpi=150, bbox_inches='tight', facecolor='white', edgecolor='none')
plt.show()
print(f'Map exported to: {png_path}')

## Step 9: Save the Updated Parcels Layer

We save separate probability scores (`probability_a1`, `probability_a2`) to the GeoPackage.
Class 6 will use these alongside consequence to calculate risk for each asset group.

In [ ]:
try:
    import fiona
    import sqlite3
    
    # Save parcels layer first (creates new GeoPackage)
    parcels.to_file(OUTPUT_GPKG, layer='parcels', driver='GPKG', mode='w')
    print(f"✓ Saved parcels layer with probability ({len(parcels)} features)")
    
    # Copy forward all other layers from the input GeoPackage
    if os.path.exists(INPUT_GPKG):
        input_layers = fiona.listlayers(INPUT_GPKG)
        for layer_name in input_layers:
            if layer_name == 'parcels':
                continue  # Already saved updated version
            try:
                layer_data = gpd.read_file(INPUT_GPKG, layer=layer_name)
                layer_data.to_file(OUTPUT_GPKG, layer=layer_name, driver='GPKG', mode='a')
                print(f"✓ Copied layer: {layer_name} ({len(layer_data)} features)")
            except Exception as e:
                print(f"  Note: Could not copy layer '{layer_name}' as spatial: {e}")
        
        # Also copy any non-spatial tables (like 'summary') via sqlite3
        try:
            conn_in = sqlite3.connect(INPUT_GPKG)
            conn_out = sqlite3.connect(OUTPUT_GPKG)
            cursor = conn_in.cursor()
            # Get all tables that aren't in fiona's layer list and aren't system tables
            cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
            all_tables = [row[0] for row in cursor.fetchall()]
            system_tables = ['gpkg_contents', 'gpkg_geometry_columns', 'gpkg_spatial_ref_sys',
                            'gpkg_ogr_contents', 'gpkg_tile_matrix', 'gpkg_tile_matrix_set',
                            'sqlite_sequence', 'gpkg_extensions', 'gpkg_metadata',
                            'gpkg_metadata_reference']
            for table in all_tables:
                if table in system_tables or table in input_layers or table.startswith('rtree_') or table.startswith('trigger_'):
                    continue
                try:
                    df = pd.read_sql(f'SELECT * FROM "{table}"', conn_in)
                    if len(df) > 0:
                        df.to_sql(table, conn_out, if_exists='replace', index=False)
                        print(f"✓ Copied non-spatial table: {table} ({len(df)} rows)")
                except Exception:
                    pass
            conn_in.close()
            conn_out.close()
        except Exception:
            pass
    
    # Also ensure base layers from Class 0 are included (flood_zones, buildings, study_area)
    # These may not be in INPUT_GPKG if earlier classes didn't carry them forward
    if os.path.exists(CLASS0_GPKG):
        try:
            import fiona as _fiona
            # Get layers already written to output
            output_layers = _fiona.listlayers(OUTPUT_GPKG)
            # Get layers available in Class 0
            class0_layers = _fiona.listlayers(CLASS0_GPKG)
            # Copy any missing layers
            for layer_name in class0_layers:
                if layer_name not in output_layers:
                    try:
                        layer_data = gpd.read_file(CLASS0_GPKG, layer=layer_name)
                        layer_data.to_file(OUTPUT_GPKG, layer=layer_name, driver='GPKG', mode='a')
                        print(f"✓ Added base layer from Class 0: {layer_name} ({len(layer_data)} features)")
                    except Exception as e:
                        print(f"  Note: Could not copy base layer '{layer_name}': {e}")
        except Exception:
            pass
    
    print(f"\n✓ All data saved to: {OUTPUT_GPKG}")
except Exception as e:
    print(f"✗ Error saving to GeoPackage: {e}")
    print("\nAlternative: Save to a new GeoPackage file")
    alt_path = os.path.join(DATA_DIR, 'class_5_probability_backup.gpkg')
    parcels.to_file(alt_path, layer='parcels', driver='GPKG')
    print(f"✓ Saved to: {alt_path}")

## QGIS Equivalent: How to Score Probability in QGIS

If you were doing this analysis in QGIS instead of Python, here's the step-by-step approach:

### Step 1: Load Layers
1. Open QGIS
2. Load the `parcels` layer from your GeoPackage
3. Load the `flood_zones` layer

### Step 2: Create the Probability Field
1. Right-click the `parcels` layer → **Edit** → enable editing
2. Right-click the layer → **Add Field**
3. Name: `probability_a1` / `probability_a2` | Type: `Integer`
4. Click OK

### Step 3: Score by Flood Zone (Using Select by Location)

**For Floodway (Score = 3):**
1. In the `flood_zones` layer, filter to show only **Floodway** features
   - Right-click flood_zones → Filter → `flood_category = 'Floodway'`
2. Click **Vector → Research Tools → Select by Location**
3. Select from: `parcels`
4. Predicate: **intersect**
5. Reference layer: `flood_zones`
6. Click Run — this selects all parcels touching the floodway
7. Open the attribute table for `parcels` (right-click → Open Attribute Table)
8. In the `probability_a1` / `probability_a2` column, right-click the header → **Field Calculator**
9. Expression: `3`
10. Click OK to apply to all selected parcels

**For 100-year Floodplain (Score = 2):**
1. Clear the current selection: **Select → Deselect All**
2. In the `flood_zones` filter, change to `flood_category = '100-year'`
3. Repeat steps 2-9 above, but:
   - In the Field Calculator, use the expression: `CASE WHEN "probability_a1" IS NULL OR "probability_a1" = 0 THEN 2 ELSE "probability_a1" END`
   - This avoids overwriting parcels already scored as 3 (Floodway)

**For 500-year Floodplain (Score = 1):**
1. Clear selection: **Select → Deselect All**
2. In the `flood_zones` filter, change to `flood_category = '500-year'`
3. Repeat steps 2-9 above, but:
   - In the Field Calculator, use: `CASE WHEN "probability_a1" IS NULL OR "probability_a1" = 0 THEN 1 ELSE "probability_a1" END`

**For Remaining Parcels (Score = 0):**
1. Clear selection: **Select → Deselect All**
2. Click **Select → By Expression**
3. Expression: `"probability_a1" IS NULL`
4. Click Select
5. Open attribute table and Field Calculator for `probability_a1` / `probability_a2`
6. Expression: `0`

### Why This Process?
QGIS doesn't have a "maximum intersection" function built-in, so we manually apply the hierarchy by processing zones in priority order (highest to lowest). By checking `"probability_a1" IS NULL` or `"probability_a1" = 0` before assigning new values, we ensure higher-priority zones don't get overwritten.

> **Note:** You will need to repeat these steps for each asset group. The field names above use the Asset 1 suffix (`_a1`). For Asset 2, replace `_a1` with `_a2` in all field references.

## ArcGIS Pro Equivalent: How to Score Probability in ArcGIS Pro

ArcGIS Pro has more automated tools for this task.

### Method 1: Using Spatial Join (Recommended)

1. In the Contents pane, right-click `parcels` → **Joins and Relates → Spatial Join**
2. Target layer: `parcels`
3. Join layer: `flood_zones`
4. Join type: **KEEP ALL TARGET FEATURES** (to keep parcels not in any zone)
5. Click **Match Option**: **INTERSECT**
6. Under Field Merge Rule, for `flood_category`:
   - Select the field `flood_category`
   - Change **Merge Rule** to **MAXIMUM**
   - (This keeps the "highest" category alphabetically, but you'll need to handle this manually)
7. Click OK

This creates a new output layer with all parcel data plus the joined flood zone information.

### Method 2: Using Select by Location + Field Calculator (Manual)

1. Right-click `parcels` → **Fields** → **Add Field**
2. Name: `probability_a1` / `probability_a2` | Type: `Short Integer`

**For Floodway (Score = 3):**
1. Use **Map → Select by Location**
2. Select features in `parcels` that **intersect** `flood_zones`
3. In the `flood_zones` filter, keep only **Floodway** visible
4. Right-click `parcels` → **Attribute Table** → Add column header context → **Field Calculator**
5. Field: `probability_a1` / `probability_a2`
6. Expression: `3`
7. Click OK

**For 100-year (Score = 2):**
1. **Select → Deselect All** to clear the selection
2. Filter `flood_zones` to show only `flood_category = '100-year'`
3. **Map → Select by Location** again with the filtered zones
4. Field Calculator with expression: `2` (but only for parcels where probability IS NULL)
   - More precisely: `IIf(IsNull(!probability!), 2, !probability!)`

**For 500-year (Score = 1):**
1. Repeat as above with expression: `1` (for null probability values)

**For No Risk (Score = 0):**
1. Select all records where `probability_a1` / `probability_a2` IS NULL
2. Field Calculator: `0`

### Alternative: Python in ArcGIS Pro

You can also write a Python script in ArcGIS Pro's **Python Window**:

```python
# Load the parcels layer
parcels = 'parcels'
flood_zones = 'flood_zones'

# Perform spatial join
arcpy.analysis.SpatialJoin(
    target_features=parcels,
    join_features=flood_zones,
    out_feature_class='parcels_joined',
    match_option='INTERSECT'
)

# Use Field Calculator to create probability based on flood_category
arcpy.management.CalculateField(
    in_table='parcels_joined',
    field='probability_a1',
    expression='3 if !flood_category! == "Floodway" else (2 if !flood_category! == "100-year" else (1 if !flood_category! == "500-year" else 0))',
    expression_type='PYTHON3'
)
```

> **Note:** You will need to repeat these steps for each asset group. The field names above use the Asset 1 suffix (`_a1`). For Asset 2, replace `_a1` with `_a2` in all field references.

## Summary: What We Accomplished

In this class, we scored **flood probability separately for Asset 1 and Asset 2**.

**Columns added:** `probability_a1`, `probability_a2`

**Scoring:** Based on the highest-priority flood zone each parcel intersects:
- 3 = Floodway (highest probability)
- 2 = 100-year floodplain
- 1 = 500-year floodplain
- 0 = Not in any flood zone

**Next:** Class 6 (Consequence) will score the financial impact separately for each asset group,
using separate median structure values as thresholds.